[← Back to 04 - JOINs](<../04-joins.ipynb>) · [Course Overview](<../00-course-overview.ipynb>)

# 04 - JOINs: Solutions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samietex/SQL_Tutorial/blob/main/solutions/04-joins-solutions.ipynb)

Try each exercise in the lesson notebook first. If your query differs from the one here but returns the same rows, it may well be just as correct, and seeing why is a useful exercise in itself.

In [1]:
# Run this cell first. It downloads the small helper that lets the cells below run SQL,
# fetching a fresh copy every time so you always have the latest version.
import os, sys, tempfile, urllib.request
helper_dir = os.path.join(tempfile.gettempdir(), 'sql_tutorial_helper')
os.makedirs(helper_dir, exist_ok=True)
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/samietex/SQL_Tutorial/assets/sql_tutorial.py',
    os.path.join(helper_dir, 'sql_tutorial.py'),
)
sys.path.insert(0, helper_dir)
%reload_ext sql_tutorial

Ready. %%sql runs SQLite 3.50.4 in memory, %%tsql runs SQL Server.


In [2]:
%%sql
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    credit_limit INTEGER NOT NULL
);
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    amount INTEGER NOT NULL
);
INSERT INTO customers VALUES
    (1, 'Ada', 100),
    (2, 'Bola', 200),
    (3, 'Chidi', 300),
    (4, 'Dami', 200);
INSERT INTO orders VALUES
    (101, 1, 40),
    (102, 1, 60),
    (103, 2, 80);

4 rows inserted
3 rows inserted


## 1. Customers with no orders

In [3]:
%%sql
SELECT
    c.customer_id,
    c.customer_name
FROM customers AS c
LEFT JOIN orders AS o
    ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL
ORDER BY
    c.customer_id;

customer_id,customer_name
3,Chidi
4,Dami


The left join keeps every customer, and `WHERE o.order_id IS NULL` keeps only those whose order columns came back empty. This pattern has a name, an **anti-join**, and it's one of the most common interview questions there is.

Use `IS NULL`, never `= NULL`: a comparison with `NULL` is never true, so `= NULL` returns no rows at all, without an error. And test a column that can't be `NULL` in a real match, like the order's primary key, so an empty result really means "no order" rather than "an order with a blank field".

## 2. Big orders per customer, including zero

In [4]:
%%sql
SELECT
    c.customer_id,
    c.customer_name,
    COUNT(o.order_id) AS big_orders
FROM customers AS c
LEFT JOIN orders AS o
    ON c.customer_id = o.customer_id
   AND o.amount >= 60
GROUP BY
    c.customer_id,
    c.customer_name
ORDER BY
    c.customer_id;

customer_id,customer_name,big_orders
1,Ada,1
2,Bola,1
3,Chidi,0
4,Dami,0


Ada has orders of 40 and 60, so only one counts. The amount condition sits in `ON`, so customers with no qualifying order keep their row, and `COUNT(o.order_id)` turns that `NULL` row into a `0`. Put the condition in `WHERE` instead and Chidi and Dami vanish, and with `COUNT(*)` they'd show `1` instead of `0`.

## 3. Stretch: products never sold (AdventureWorks)

This answer runs on SQL Server, so set up AdventureWorks first. In Colab that takes a few minutes the first time, and on your own machine it only checks the connection.

In [5]:
# Run this once before any %%tsql cell.
# In Google Colab it installs SQL Server and AdventureWorks into this session (about 3 to 5
# minutes the first time, and again in each new session, because Colab starts fresh).
# On your own machine it just checks that your SQL Server is reachable.
%setup_adventureworks

AdventureWorks2022 is ready, so every %%tsql cell in this notebook will run.


In [6]:
%%tsql
SELECT COUNT(*) AS never_sold
FROM Production.Product AS p
LEFT JOIN Sales.SalesOrderDetail AS d
    ON p.ProductID = d.ProductID
WHERE d.SalesOrderDetailID IS NULL;

never_sold
238


238 of the 504 products have never appeared on an order. It's the same anti-join as exercise 1, and it's also a good place to see why the grain question matters: a product that *was* sold appears once per order line in the joined result, so `COUNT(*)` without the `IS NULL` filter would count order lines, not products. To see which products they are:

In [7]:
%%tsql
SELECT TOP 10
    p.ProductID,
    p.Name
FROM Production.Product AS p
LEFT JOIN Sales.SalesOrderDetail AS d
    ON p.ProductID = d.ProductID
WHERE d.SalesOrderDetailID IS NULL
ORDER BY
    p.ProductID;

ProductID,Name
1,Adjustable Race
2,Bearing Ball
3,BB Ball Bearing
4,Headset Ball Bearings
316,Blade
317,LL Crankarm
318,ML Crankarm
319,HL Crankarm
320,Chainring Bolts
321,Chainring Nut


---

[← Back to 04 - JOINs](<../04-joins.ipynb>) · [Course Overview](<../00-course-overview.ipynb>)

*SQL_Tutorial* is written and maintained by Samuel Shaibu as part of *All About Data & More*. Licensed under [MIT](../LICENSE).